In [1]:
import time
from time import sleep
import numpy as np
from ms2k import MS2000
from TDI import TDI
from obis_laser_controller import ObisLaserController
import os
from RT_infer import RTInfer
from pathlib import Path
import imageio
import cv2

def main():
    # Initialize laser
    laser = ObisLaserController( 'COM6' )
    laser.connect()
    # set power
    laser.set_power( power = 0.001 ) # in Watts # max pawer is 0.066 W
 
    # Initialize TDI
    tdi = TDI()
    if not tdi.initialize_grabber():
        print("Failed to initialize TDI grabber. Exiting.")
        return
    
    # Load calibration image
    folder_path = "output/260116_mouse_kidney_2ch/calibration"
    cal_profile = tdi.average_column_profile(folder_path, 20)
        
    # Initialize XY stage (MS2000 stage)
    print(f"COM Ports: {MS2000.scan_ports()}")
    ms2k = MS2000("COM5", 9600)
    ms2k.connect_to_serial()
    if not ms2k.is_open():
        print("Exiting the program...")
        return

    # Parameters
    x_speed = 0.074
    x_speed_return = 0.074*32
    y_speed = 0.074*32 
    ms2k.set_max_speed("Y", y_speed)
    frame_num = 30 # Number of frames to capture per row
    x_move_distance = 80*10*frame_num  # Move distance for the X stage in microns (+: stage left, -: stage rights)
    y_move_distance = -60*10 # Move Y stage (+: stage up, -: stage down)
    row_num = 35
    num_repeats = 1 # Number of repeated acquisitions
    time_interval = 0 # in seconds
    crop_info = {'crop_width': 1024, 'center_col': 2242} # Crop info
    # crop_info = {'crop_width': 1024*2, 'center_col': 2242-1024} # Crop info WF (move y stage to -0.082 mm)
        
    # Output path for TDI images
    output_path = "output/260116_mouse_kidney_2ch/01_488nm"  
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    print(f"Saving all images to: {output_path}")
        
    # Initialize RTInfer with dummy image
    # model_weights_dir = Path('C:/Users/Nikon_1/Documents/JupyterLab/hansol/ZS-DeconvNet/Python_MATLAB_Codes/your_saved_models/ellipses4000_with_Gaussian0p5sigma_TDInoise3sigma_twostage_Unet_Hess0.02/weights_6000.h5')
    model_weights_dir = Path('C:/Users/Nikon_1/Documents/JupyterLab/hansol/ZS-DeconvNet/Python_MATLAB_Codes/your_saved_models/ellipses4000_with_100filaments_with_Gaussian0p5sigma_TDInoise3sigma_twostage_Unet_Hess0.02/weights_6000.h5')
    dummy_image = np.random.rand(1024, 1024).astype(np.float32) * 4095  # simulate 12-bit data
    infer_runner = RTInfer(model_weights_path=model_weights_dir, raw_data=dummy_image)    
    
    # xy scanning
    all_frames_xy = [] # Store all captured frames
    initial_x = ms2k.get_position("X")
    initial_y = ms2k.get_position("Y")
    print(f"Initial XY position: X={initial_x} µm, Y={initial_y} µm")

    start_time = None  # Will store the start time at first capture
    
    capture_idx = 0
    start_time = None
    all_frames_xy = []

    for i in range(num_repeats):
        print(f"\n=== Temporal Repeat {i + 1} of {num_repeats} ===")

        for row_idx in range(row_num):
            print(f"\n--- Row {row_idx + 1} of {row_num} ---")

            laser.on()

            # Move X stage
            ms2k.set_max_speed("X", x_speed)
            ms2k.moverel_axis("X", x_move_distance)
            print(f"Started moving X stage to {x_move_distance} µm")

            if start_time is None:
                start_time = time.time()

            frames = tdi.capture_frames_to_memory_live(
                max_frames=frame_num,
                display_zoom=0.5,
                num_buffers=3,
                cal_flag=1,
                cal_profile=cal_profile,
                crop_info=crop_info,
                infer_flag=1,
                infer_runner=infer_runner
            )

            timestamp_ms = int((time.time() - start_time) * 1000)
            print(f"Timestamp after capture: {timestamp_ms} ms")

            capture_idx += 1
            all_frames_xy.append((i + 1, row_idx + 1, frames))  # Store temporal repeat, row index, and data

            laser.off()

            # Wait and return X
            ms2k.wait_for_device()
            print(f"Moved X stage to {x_move_distance} µm")

            ms2k.set_max_speed("X", x_speed_return)
            ms2k.moverel(-x_move_distance, 0)
            ms2k.wait_for_device()
            print("Returned X stage to original position")

            # Move Y to next row (unless last row)
            if row_idx < row_num - 1:
                ms2k.moverel(0, y_move_distance)
                ms2k.wait_for_device()
                print(f"Moved Y stage by {y_move_distance} µm")

        # Return Y to initial position after one full scan
        ms2k.moverel(0, -y_move_distance * (row_num - 1))
        ms2k.wait_for_device()
        print("Returned Y stage to initial position")
        time.sleep(time_interval)
    
    cv2.destroyAllWindows()  # Clean up at the very end

    # Save images
    for repeat_idx, row_idx, frames in all_frames_xy:
        subdir = os.path.join(output_path, f"t_{repeat_idx:03d}", f"row_{row_idx:03d}")
        os.makedirs(subdir, exist_ok=True)
        tdi.save_images_to_disk(frames, outdir=subdir, filename_prefix="frame", keys_to_save=['raw', 'calibrated', 'denoise', 'decon'])

    print(f"\nAll {capture_idx} captures completed and saved.")

    # Close the connections
    ms2k.disconnect_from_serial()
    laser.disconnect()


if __name__ == "__main__":
    main()


TensorFlow version: 2.5.0
Built with CUDA: True
GPU devices available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: 64_112
Interface:    PC3622 - Coaxlink Duo CXP-12 (1-camera, line-scan) - KDP01002
Device:       Device0
Resolution:   4640x1
Grabber successfully initialized.
COM Ports: ['COM4', 'COM5', 'COM6', 'COM7', 'COM9']
Connected to the serial port.
Serial port = COM5 :: Baud rate = 9600
Send: SPEED Y=2.368
Recv: :A
Saving all images to: output/260303_buccal_SPI_WF/01_WF
Initializing model...
Running dummy inference to warm up model...
[DEBUG] Raw image stats before normalization: min=0.003935329150408506, max=4094.994873046875, mean=2047.2423095703125
Input image shape before padding: (1024, 1024)
1/1 [==============================] - 18s 18s/step
Initialization complete.
Send: WHERE X
Recv: :A 0
Send: WHERE Y
Recv: :A -817
Initial XY position: X=0 µm, Y=-817 µm

=== Temporal Repeat 1 of 1 ===

--- Row 1 of 1 ---
Send: SPEED X=0.074
Rec